In [20]:
%load_ext autoreload
%autoreload 2

# 3W Windowed Dataset: Exploration

**Input:** `data/gold/3w_windows_real.parquet`, built by `assetmind.pipelines.build_3w_windows`.
5-minute windows with a 1-minute step over all 1,119 real instances; sensors P-TPT, T-TPT, P-ANULAR; each window labeled by its last second.

**Goal:** check the label and phase distribution before building the anomaly detector.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
w = pd.read_parquet(ROOT / "data/gold/3w_windows_real.parquet")
print(w.shape)
w["phase"].value_counts()

(475682, 25)


phase
normal       239490
transient    183383
event         52809
Name: count, dtype: int64

In [2]:
pd.crosstab(w["event_class"], w["phase"])

phase,event,normal,transient
event_class,,,
0,0,164319,0
1,163,2436,1090
2,356,1359,1485
3,9342,0,0
4,39353,0,0
5,221,1051,5342
6,223,574,108
7,507,17080,111178
8,2349,15977,60923


In [3]:
rep = pd.read_parquet(ROOT / "reports/3w_lowo_baseline.parquet")
ev = rep[rep["has_event"]].assign(detected=lambda d: d["detected"].astype(bool))

ev.groupby("event_class").agg(
    files=("instance_id", "size"),
    wells=("well", "nunique"),
    detected=("detected", "mean"),
    median_delay_h=("delay_h", "median"),
    median_lead_h=("lead_time_h", "median"),
    scored=("scored_share", "mean"),
).round(2)

,files,wells,detected,median_delay_h,median_lead_h,scored
event_class,,,,,,
1,4,3,0.00,NaN,NaN,0.95
2,22,7,1.00,0.05,1.38,0.99
3,32,2,0.84,0.10,-0.10,1.00
4,343,7,0.01,0.85,-0.85,1.00
5,11,3,0.55,0.05,0.78,1.00
6,6,2,0.00,NaN,NaN,0.98
7,36,6,0.00,NaN,NaN,0.88
8,14,9,0.64,0.12,32.22,0.92
9,14,7,0.07,0.05,0.13,0.97


## Baseline detector results (robust z on std + change, q = 0.995, k = 3, leave-one-well-out)

- Caught well: DHSV closure (100%, 3 min, +1.4 h lead), slugging (84%, 6 min), productivity loss (55%), hydrate in production line (64%, +32 h lead; onset may coincide with an operational disturbance, to verify).
- Missed: BSW increase (0%), quick PCK restriction (0%), scaling (0%), hydrate in service line (7%). These are level shifts and slow drifts, invisible to movement features in 5-min windows.
- Flow instability: 1% detected, despite being an oscillation; it is 343 of 525 event files, so it dominates the overall rate. Under investigation.
- Transient-type events caught before the event set in: 24%; with ≥1 h warning: 17%.
- False alarms: 0.79 per well per day.
- Class 9: only 14 of 57 real files contain abnormal windows. To verify.
- Metric note: the warning-time curve is computed only for event types with a transient phase; slugging and instability can never have positive lead time by definition.

In [4]:
import numpy as np
import plotly.express as px

w["phase"] = w["phase"].astype(str)
sub = w[(w["phase"] == "normal") | w["event_class"].isin([3, 4])].copy()
sub["group"] = np.where(sub["phase"] == "normal", "normal",
                        "class " + sub["event_class"].astype(str))
sub["log_std_P-TPT"] = np.log1p(sub["P-TPT_std"])

print(sub.groupby("group")["P-TPT_std"].median().round(0))
px.box(sub, x="group", y="log_std_P-TPT", title="P-TPT movement: normal vs slugging vs flow instability").show()

group
class 3    57006.0
class 4    14836.0
normal      5240.0
Name: P-TPT_std, dtype: float64


In [5]:
c9 = (w[w["event_class"] == 9]
      .groupby("instance_id", observed=True)["phase"]
      .apply(lambda s: (s != "normal").any()))
print("Class 9 files with abnormal windows:", int(c9.sum()), "of", len(c9))


Class 9 files with abnormal windows: 14 of 57


### Box plot: P-TPT movement (log std)

- Slugging's middle 50% sits entirely above normal's; it is separated well.
- Flow instability's middle 50% sits above normal's box but inside normal's upper whisker, so it overlaps with busy normal operation.
- Normal ranges from near-zero movement to values exceeding slugging (log std up to ~15). The extreme "normal" windows are likely planned operations (shut-ins, restarts, large choke changes): labeled normal, but extreme in sensor terms.
- These extreme normal windows push the 99.5th-percentile threshold up, hiding moderate events like instability.

**Ideas**
- Per-well baselines (pending the per-well comparison).
- Operational context from valve states to recognize planned operations.

In [6]:
W3 = ROOT.parent / "3W" / "dataset"
no_event = c9[~c9].index[:5]

for inst in no_event:
    cls, stem = str(inst).split("/")
    raw = pd.read_parquet(W3 / cls / f"{stem}.parquet", columns=["class"])
    print(stem, raw["class"].value_counts(dropna=False).to_dict())

WELL-00010_20180517000402 {np.int16(0): 73122, <NA>: 3600}
WELL-00010_20180715083053 {np.int16(0): 16094, <NA>: 3600}
WELL-00010_20180717234051 {np.int16(0): 15396, <NA>: 3600}
WELL-00010_20180720234103 {np.int16(0): 19074, <NA>: 3600}
WELL-00010_20180723134436 {np.int16(0): 83448, <NA>: 3600}


### Class 9 folder contents

Only 14 of 57 real files in the class 9 folder contain abnormal labels. Raw-file check: the other files contain only normal (0) and unlabeled rows, e.g. five files from WELL-00010. Not a pipeline issue.

- These files act as normal-only instances in evaluation (they count toward false alarms, not detection).
- Class 9 results are reported on the 14 labeled event instances (7 wells).
- Possible label noise: unlabeled early hydrate formation could exist in these "normal" windows.

In [7]:
normal_by_well = w[w["phase"] == "normal"].groupby("well", observed=True)["P-TPT_std"].median()
c4_by_well = w[w["event_class"] == 4].groupby("well", observed=True)["P-TPT_std"].median()

compare = pd.DataFrame({"normal_median": normal_by_well,
                        "instability_median": c4_by_well}).round(0)
compare.sort_values("normal_median")

,normal_median,instability_median
well,,
WELL-00014,33.0,11247.0
WELL-00032,180.0,NaN
WELL-00021,736.0,NaN
WELL-00013,782.0,NaN
WELL-00003,816.0,NaN
WELL-00012,968.0,NaN
WELL-00011,1284.0,NaN
WELL-00027,1603.0,NaN
WELL-00005,1985.0,18674.0


### Per-well normal movement vs instability

- Normal P-TPT std varies ~1,400× between wells (33 Pa for WELL-00014 to 46,031 Pa for WELL-00025). A pooled normal profile is too wide to detect moderate events.
- In all 6 instability wells with valid data, instability moves more than that well's own normal: ~340× (WELL-00014), ~9× (00005, 00007), ~5× (00001), but only ~2× (00002) and ~1.4× (00004).
- 14 wells have no usable normal P-TPT data, so a per-well baseline is impossible for them; the 7th instability well has no valid P-TPT.

**Decision: two-tier detector (v2)**
- Wells with earlier normal history: score movement relative to the well's own baseline.
- Wells without history: fall back to the global detector.
- No-future rule: an instance's baseline may only use that well's normal files that ended before the instance began.
- Expected limits: wells 00002 and 00004 (small ratios) and wells without history.

In [8]:
from assetmind.evaluation.event_metrics import false_alarms_per_day

v1 = pd.read_parquet(ROOT / "reports/3w_lowo_baseline.parquet")
v2 = pd.read_parquet(ROOT / "reports/3w_lowo_twotier.parquet")

def detected_by_class(rep):
    ev = rep[rep["has_event"]].assign(detected=lambda d: d["detected"].astype(bool))
    return ev.groupby("event_class")["detected"].mean()

ev2 = v2[v2["has_event"]]
compare_v = pd.DataFrame({
    "files": ev2.groupby("event_class").size(),
    "v1_detected": detected_by_class(v1),
    "v2_detected": detected_by_class(v2),
    "v2_share_well_tier": ev2.groupby("event_class")["tier"].apply(lambda s: (s == "well").mean()),
}).round(2)
print(compare_v.to_string())

print("\nTier used, all instances:", v2["tier"].value_counts().to_dict())
for tier, g in v2.groupby("tier"):
    print(f"False alarms per day, {tier} tier: {false_alarms_per_day(g):.2f}")

             files  v1_detected  v2_detected  v2_share_well_tier
event_class                                                     
1                4         0.00         0.00                0.75
2               22         1.00         1.00                0.59
3               32         0.84         0.97                1.00
4              343         0.01         0.08                0.63
5               11         0.55         0.55                0.45
6                6         0.00         0.50                1.00
7               36         0.00         0.00                0.78
8               14         0.64         0.71                0.36
9               14         0.07         0.21                0.43

Tier used, all instances: {'well': 859, 'global': 260}
False alarms per day, global tier: 3.02
False alarms per day, well tier: 0.48


In [9]:
both = v1[["instance_id", "false_alarms", "normal_hours"]].merge(
    v2[["instance_id", "false_alarms", "normal_hours", "tier"]],
    on="instance_id", suffixes=("_v1", "_v2"))

for tier, g in both.groupby("tier"):
    v1_rate = g["false_alarms_v1"].sum() / (g["normal_hours_v1"].sum() / 24)
    v2_rate = g["false_alarms_v2"].sum() / (g["normal_hours_v2"].sum() / 24)
    print(f"{tier:6s} instances: v1 {v1_rate:.2f}/day  ->  v2 {v2_rate:.2f}/day")

global instances: v1 3.02/day  ->  v2 3.02/day
well   instances: v1 0.24/day  ->  v2 0.48/day


## v2: two-tier detector (per-well baseline with no-future rule, else global)

| Class | Files | v1 detected | v2 detected | v2 per-well tier share |
|---|---|---|---|---|
| 1 | 4 | 0.00 | 0.00 | 0.75 |
| 2 | 22 | 1.00 | 1.00 | 0.59 |
| 3 | 32 | 0.84 | 0.97 | 1.00 |
| 4 | 343 | 0.01 | 0.08 | 0.63 |
| 5 | 11 | 0.55 | 0.55 | 0.45 |
| 6 | 6 | 0.00 | 0.50 | 1.00 |
| 7 | 36 | 0.00 | 0.00 | 0.78 |
| 8 | 14 | 0.64 | 0.71 | 0.36 |
| 9 | 14 | 0.07 | 0.21 | 0.43 |

- Detection improved for 5 classes and worsened for none. Quick PCK restriction went from 0 to 3 of 6, as the event stands out against the well's own calm normal.
- Flow instability improved only to 8% despite the per-well tier in 63% of cases.
- Tier usage: per-well 859 instances, global 260.
- False alarms: comparing tiers directly (global 3.02/day vs per-well 0.48/day) is misleading because of a selection effect: the global tier handles each well's earliest files and wells without usable normal data. Like-for-like on the same instances: global-tier instances are unchanged (3.02/day in both versions); per-well-tier instances rose from ~0.24 to 0.48/day.
- Conclusion: v2 trades roughly double false alarms (on per-well-tier instances) for better detection. Calibrating the per-well threshold in-sample makes it too tight for later data.

**Next improvements**
- Calibrate the per-well threshold out-of-sample (baseline from earlier files, scored on later files).
- Measure the detection vs false-alarm trade-off across thresholds.
- Investigate high false alarms on global-tier instances (new and uninstrumented wells).

In [10]:
v3 = pd.read_parquet(ROOT / "reports/3w_lowo_twotier_out_of_sample.parquet")

compare_v["v3_detected"] = detected_by_class(v3).round(2)
print(compare_v[["files", "v1_detected", "v2_detected", "v3_detected"]].to_string())

both3 = both.merge(v3[["instance_id", "false_alarms", "normal_hours"]].rename(
    columns={"false_alarms": "false_alarms_v3", "normal_hours": "normal_hours_v3"}), on="instance_id")
for tier, g in both3.groupby("tier"):
    rates = {v: g[f"false_alarms_{v}"].sum() / (g[f"normal_hours_{v}"].sum() / 24) for v in ["v1", "v2", "v3"]}
    print(f"{tier:6s}: " + "  ".join(f"{v} {r:.2f}/day" for v, r in rates.items()))

             files  v1_detected  v2_detected  v3_detected
event_class                                              
1                4         0.00         0.00         0.00
2               22         1.00         1.00         1.00
3               32         0.84         0.97         0.97
4              343         0.01         0.08         0.07
5               11         0.55         0.55         0.45
6                6         0.00         0.50         0.00
7               36         0.00         0.00         0.00
8               14         0.64         0.71         0.50
9               14         0.07         0.21         0.14
global: v1 3.02/day  v2 3.02/day  v3 3.02/day
well  : v1 0.24/day  v2 0.48/day  v3 0.02/day


## v3: out-of-sample calibration of the per-well threshold

- Per-well-tier false alarms: 0.24 (v1) → 0.48 (v2) → 0.02/day (v3). Global tier unchanged at 3.02/day; it is now the main source of false alarms (overall 0.61/day).
- Detection dropped for subtler events: quick PCK restriction 50% → 0%, hydrate (production) 71% → 50%, productivity loss 55% → 45%, hydrate (service) 21% → 14%. Slugging (97%) and DHSV closure (100%) unaffected.
- v2's threshold was too tight (in-sample calibration); v3's is likely too loose: calibration compares a well's earliest half with its latest half, a larger time gap than at test time, when the baseline includes recent files.
- v2 and v3 are two points on one detection vs false-alarm trade-off curve. Next: map the full curve rather than fine-tune the calibration rule.

In [11]:
sweep = pd.read_parquet(ROOT / "reports/3w_sweep_twotier.parquet")
fig = px.scatter(sweep, x="false_alarms_per_day", y="detection_macro",
                 color=sweep["k"].astype(str), text="quantile",
                 title="Trade-off: macro detection vs false alarms per well per day",
                 labels={"color": "k (consecutive windows)"})
fig.update_traces(textposition="top center")
fig.show()

## Trade-off sweep (two-tier, out-of-sample calibration; quantile ∈ {0.99, 0.995, 0.999}, k ∈ {1, 3, 5})

- Some settings dominate others: e.g. q=0.99/k=5 (macro detection 0.45, 1.06 FA/day) beats q=0.995/k=1 (0.37, 1.34) on both. All k=1 settings are dominated.
- Pareto front: q=0.999/k=5 (0.27, 0.07), q=0.999/k=3 (0.28, 0.08), q=0.995/k=5 (0.35, 0.25), q=0.99/k=5 (0.45, 1.06), q=0.99/k=3 (0.465, 1.33).
- Persistence is nearly free: at q=0.995, k=1→5 cuts false alarms by >80% (1.34 → 0.25/day) with negligible loss in detection; warning time is unaffected.
- False alarms come mainly from the global tier (e.g. 1.22 vs 0.015/day at q=0.995/k=5), so one shared threshold suits neither tier.
- Ceiling: macro detection ≤ ~0.47 and ≥1 h warning ≤ ~0.17 at every setting. The limit is the features (movement only, 5-min windows), not the threshold.
- Default operating point: q=0.995, k=5 (macro detection 0.35, 0.25 FA/well/day).

**Next**
- Separate thresholds for the per-well and global tiers.
- Move the curve, not along it: level features, longer windows, stronger models. Any new model must beat this curve.

In [12]:
from assetmind.ingestion.threew import read_instance, make_windows, MODEL_SENSORS

W3 = ROOT.parent / "3W" / "dataset"
valid = pd.read_parquet(ROOT / "reports/3w_real_validity.parquet")
f7 = valid[(valid["class"] == 7) & (valid["P-TPT"] > 0.9)]["file"].iloc[0]   # a scaling file with good P-TPT

w7 = make_windows(read_instance(W3 / "7" / f7, MODEL_SENSORS), MODEL_SENSORS)
px.scatter(w7, x="window_end", y="P-TPT_mean", color="phase",
           title=f"Class 7 (scaling): P-TPT 5-min mean over the whole file").show()

print("Typical 5-min change during transient:", round(w7.loc[w7["phase"] == "transient", "P-TPT_change"].abs().median()))
print("Typical 5-min std during normal:      ", round(w7.loc[w7["phase"] == "normal", "P-TPT_std"].median()))

Typical 5-min change during transient: 999
Typical 5-min std during normal:       964


### Scaling (class 7) on P-TPT: one file, viewed whole

- This file's transient lasts ~6.5 h (07:10–13:40); the 51 h figure is an average across scaling files.
- P-TPT drifts up only ~0.05–0.06 bar during the transient; the whole file spans ~0.14 bar.
- Normal operation shows slow 1–2 h waves of ~0.07–0.08 bar, the same size as the scaling creep, so the signal is subtle even across hours.
- Clearest change: a ~0.06 bar step around 11:30–12:00 that persists, ~1.5–2 h before the event.
- Physical reason: scaling occurs in the production choke on the platform, while P-TPT is far upstream at the subsea tree, so the effect is diluted.

**Implications**
- Multi-scale features (longer windows, level vs the well's own baseline) should help partly, e.g. catching the persistent step.
- P-MON-CKP (upstream of the production choke, usable in ~65% of files) is likely the better sensor for scaling; test it as an extended input.

- Inside a 5-min window: typical change during the scaling transient = 999 Pa; typical normal std = 964 Pa. The two are indistinguishable.
- The creep itself (~6,000 Pa over ~6.5 h) is only ~80 Pa per 5-min window, about 1/12 of normal noise.
- Conclusion: a lower threshold cannot detect scaling earlier; the information is not in a 5-min window. The ceiling on the trade-off curve comes from the features, not the threshold.

In [13]:
dur = (w.groupby("instance_id", observed=True)
        .agg(event_class=("event_class", "first"),
             hours=("window_end", lambda s: (s.max() - s.min()).total_seconds() / 3600)))

dur.groupby("event_class")["hours"].describe(percentiles=[0.1, 0.5, 0.9]).round(1)

,count,mean,std,min,10%,50%,90%,max
event_class,,,,,,,,
0,594.0,4.6,1.6,1.8,2.9,4.9,4.9,38.7
1,4.0,15.4,9.2,3.9,7.2,15.5,23.4,26.5
2,22.0,2.4,1.6,0.9,1.1,1.8,4.2,7.4
3,32.0,4.8,0.2,3.9,4.8,4.9,4.9,4.9
4,343.0,1.9,0.1,1.8,1.9,1.9,1.9,2.8
5,11.0,10.0,11.5,0.4,0.8,7.6,16.2,40.7
6,6.0,2.5,2.4,0.5,0.5,2.0,5.0,6.7
7,36.0,59.6,48.8,5.8,13.4,45.0,134.2,212.3
8,14.0,94.3,56.5,36.2,41.0,78.0,180.1,199.5


## v4 design: choosing a time scale for slow changes

File length (hours of labeled data) by class: normal ~4.9 h, slugging ~4.9 h, instability ~1.9 h, DHSV ~1.8 h, quick restriction ~2.0 h; scaling median 45 h (13–212 h), hydrate (production) median 78 h (36–200 h). Fixed lengths for several classes suggest another dataset convention.

- 6-hour windows rejected: empty in nearly all normal, slugging, instability and DHSV files (so "normal" can't be learned), and present mostly in scaling and hydrate files, so a feature's existence would encode the class (shortcut via file length).
- "Change since file start" rejected: drift accumulates with elapsed time, so long files would look abnormal regardless of physics.
- Chosen: fixed 1-hour look-back features, identical for every file:
  - 1-h level change = current 5-min mean − 5-min mean 60 min earlier
  - 1-h waviness = std of the 5-min means over the last hour

In [14]:
from assetmind.models.eval_anomaly_twotier import load_windows, evaluate
from assetmind.evaluation.event_metrics import false_alarms_per_day

wv, meta = load_windows(ROOT / "data/gold/3w_windows_real_v4.parquet")
feature_sets = {"v3": ("logstd", "change"), "v4": ("logstd", "change", "lvl1h", "logwav1h")}
reps = {name: evaluate(wv, meta, 0.995, 5, "out_of_sample", feats) for name, feats in feature_sets.items()}

for name, rep in reps.items():
    rep.to_parquet(ROOT / f"reports/3w_lowo_{name}_q0995_k5.parquet", index=False)

per_class = pd.DataFrame({name: detected_by_class(rep) for name, rep in reps.items()}).round(2)
per_class.insert(0, "files", reps["v4"][reps["v4"]["has_event"]].groupby("event_class").size())
print(per_class.to_string())

for name, rep in reps.items():
    by_class = {c: round(false_alarms_per_day(g), 2) for c, g in rep.groupby("event_class")}
    print(f"\n{name} false alarms/day by source folder:", by_class)

             files    v3    v4
event_class                   
1                4  0.00  0.00
2               22  1.00  1.00
3               32  0.97  0.97
4              343  0.07  0.06
5               11  0.45  0.45
6                6  0.00  0.00
7               36  0.00  0.00
8               14  0.50  0.50
9               14  0.14  0.14

v3 false alarms/day by source folder: {0: 0.0, 1: 0.0, 2: 0.0, 3: nan, 4: nan, 5: 16.44, 6: 0.0, 7: 0.0, 8: 1.8, 9: 0.39}

v4 false alarms/day by source folder: {0: 0.04, 1: 0.0, 2: 0.0, 3: nan, 4: nan, 5: 1.37, 6: 0.0, 7: 0.0, 8: 0.09, 9: 0.24}


## v4: fixed 1-hour look-back features (lvl1h, wav1h)

- Coverage: lvl1h exists in ~49–56% of windows, wav1h in ~39–45% (no look-back in each file's first hour; invalid windows break the hour). The 1-minute spacing check passed for all files.
- At equal false alarms (0.07/well/day), v4 detects 0.35 vs v3's 0.27 (macro). At equal detection (0.35), v4 has 0.07 vs v3's 0.25 false alarms/day. The curve moved up and to the left.
- Gain comes mainly from fewer false alarms (global tier 1.22 → 0.18/day at q=0.995); per-well tier rose slightly (0.015 → 0.045).
- Early warning (≥1 h) is unchanged (0.14 at q=0.995): the new features did not help slow events as intended. Hypothesis: they help separate brief disturbances from sustained shifts.

In [15]:
import importlib
import assetmind.evaluation.event_metrics as em
import assetmind.models.eval_anomaly_twotier as ev
importlib.reload(em); importlib.reload(ev)   # pick up the edited code without restarting

rep_v3 = ev.evaluate(wv, meta, 0.995, 5, "out_of_sample", ("logstd", "change"))
fa = rep_v3[rep_v3["has_event"]].explode("fa_hours_before_onset").dropna(subset=["fa_hours_before_onset"])
fa["hours_before"] = fa["fa_hours_before_onset"].astype(float)

print("False alarm episodes in event files:", len(fa))
for h in [0.5, 1, 2, 4]:
    print(f"  within {h} h before onset: {(fa['hours_before'] <= h).mean():.0%}")
print(fa.groupby("event_class")["hours_before"].describe().round(2))
px.histogram(fa, x="hours_before", color=fa["event_class"].astype(str), nbins=40,
             title="When do 'false alarms' in event files happen? (hours before labeled onset)").show()

False alarm episodes in event files: 34
  within 0.5 h before onset: 12%
  within 1 h before onset: 24%
  within 2 h before onset: 41%
  within 4 h before onset: 65%
             count  mean   std   min   25%   50%   75%    max
event_class                                                  
5             12.0  1.39  0.91  0.13  0.70  1.30  2.07   2.92
8             20.0  5.69  3.93  0.15  2.12  5.98  9.60  10.63
9              2.0  4.24  0.51  3.88  4.06  4.24  4.42   4.60


### v3 vs v4 per class and false-alarm sources (q=0.995, k=5)

- Detection identical per class; v4's gain comes entirely from fewer false alarms. No slow event improved.
- Pure normal files produced 0.00 false alarms/day in v3 (0.04 in v4). Nearly all false alarms come from the normal periods inside event files: productivity loss 16.44/day (v3) → 1.37 (v4), hydrate (production) 1.80 → 0.09, hydrate (service) 0.39 → 0.24.
- The false-alarm metric is therefore dominated by alarms in the hours before labeled events, not by noise in normal operation.
- Two explanations: (1) genuine early warnings, with expert labels starting after the physical onset; (2) genuine disturbances before events. Testing by timing each false alarm relative to the onset.

### Timing of "false alarms" in event files (v3, q=0.995, k=5)

- Only 34 false-alarm episodes in event files; the 16/day for productivity loss came from 12 episodes over little normal time (small denominator).
- Productivity loss (5): all 12 episodes 0.1–2.9 h before onset (median 1.3 h). Hydrate (8): spread 0.2–10.6 h (median 6.0 h). Hydrate (9): 2 episodes.
- Overall: 24% within 1 h, 41% within 2 h, 65% within 4 h of the labeled onset.
- Caveat: alarms can only occur in the normal period, which ends at the onset, so short normal periods force alarms close to the onset. Checking against a uniform (random) baseline using each alarm's relative position in its file's normal period.

In [16]:
normal_h = (wv[wv["phase"] == "normal"].groupby("instance_id").size() / 60).rename("normal_hours_file")
fa2 = fa.merge(normal_h, left_on="instance_id", right_index=True)
fa2["relative_position"] = fa2["hours_before"] / fa2["normal_hours_file"]

print(fa2.groupby("event_class")[["normal_hours_file", "relative_position"]].median().round(2))
print("\nShare in the last quarter of the normal period (random ≈ 25%):",
      f"{(fa2['relative_position'] <= 0.25).mean():.0%}")
px.histogram(fa2, x="relative_position", color=fa2["event_class"].astype(str), nbins=10,
             range_x=[0, 1], title="Alarm position in the normal period (0 = at onset, 1 = start)").show()

             normal_hours_file  relative_position
event_class                                      
5                         3.10               0.42
8                        12.05               0.50
9                         4.31               0.98

Share in the last quarter of the normal period (random ≈ 25%): 35%


### Are false alarms in event files early warnings? (null-baseline check)

- Relative position of each false alarm within its file's normal period (0 = at onset, 1 = start); random placement would give a uniform spread (median 0.5, 25% in the last quarter).
- Productivity loss (5): normal periods are short (median 3.1 h), which alone forces alarms close to the onset; relative median 0.42, consistent with chance. The apparent 1–3 h "early warning" was an artifact.
- Hydrate (8): relative median 0.50, consistent with chance. Hydrate (9): 2 episodes at ~0.98.
- Overall: 35% in the last quarter vs 25% expected (~12 vs ~8.5 of 34; ~1.4 SD, episodes not independent). At most a weak hint; most are genuine false alarms.
- Pure normal files: ~0 false alarms/day; normal periods inside event files: nearly all false alarms. Hypothesis: the normal folder contains curated calm periods (fixed 4.9 h length), so the blended false-alarm rate is optimistic.

**Decision:** report false alarms separately for (a) normal files and (b) normal periods inside event files.

### Consequence of curated normal data

If normal files are curated calm periods, the learned normal range is too narrow: real operation (with routine disturbances) triggers more false alarms, and keeping alarms acceptable would require a higher threshold, lowering detection. The measured trade-off curve is likely optimistic on both axes.

Future experiment: learn normal only from unselected normal periods inside event files and compare curves.

In [17]:
ext = "P-MON-CKP"
rows = []
for path in sorted(W3.glob("[0-9]/WELL-*.parquet")):
    wx = make_windows(read_instance(path, [ext]), [ext])
    rows.append({"class": int(path.parent.name), "file": path.name,
                 "ok_share": float((wx[f"{ext}_status"] == "ok").mean())})

vx = pd.DataFrame(rows)
vx.to_parquet(ROOT / "reports/3w_validity_pmonckp.parquet", index=False)
vx["usable"] = vx["ok_share"] >= 0.5

print(f"{ext} usable in {vx['usable'].mean():.0%} of real files")
vx.groupby("class")["usable"].agg(usable_files="sum", files="size", share="mean").round(2)

P-MON-CKP usable in 64% of real files


,usable_files,files,share
class,,,
0,326,594,0.55
1,3,4,0.75
2,7,22,0.32
3,32,32,1.00
4,231,343,0.67
5,11,11,1.00
6,3,6,0.50
7,36,36,1.00
8,14,14,1.00


In [18]:
f7x = vx[(vx["class"] == 7) & (vx["ok_share"] > 0.9)]["file"].iloc[0]
both = make_windows(read_instance(W3 / "7" / f7x, ["P-TPT", ext]), ["P-TPT", ext])
for s in ["P-TPT", ext]:
    px.scatter(both, x="window_end", y=f"{s}_mean", color="phase",
               title=f"Class 7 (scaling), {f7x}: 5-min mean of {s}").show()

### Extended sensor P-MON-CKP (upstream of the production choke): usability

Usable (ok in ≥ 50% of windows) by class:

| Class | Usable files | Share |
|---|---|---|
| 0 Normal | 326 / 594 | 0.55 |
| 1 BSW increase | 3 / 4 | 0.75 |
| 2 DHSV closure | 7 / 22 | 0.32 |
| 3 Slugging | 32 / 32 | 1.00 |
| 4 Flow instability | 231 / 343 | 0.67 |
| 5 Productivity loss | 11 / 11 | 1.00 |
| 6 Quick restriction | 3 / 6 | 0.50 |
| 7 Scaling | 36 / 36 | 1.00 |
| 8 Hydrate (production) | 14 / 14 | 1.00 |
| 9 Hydrate (service) | 57 / 57 | 1.00 |

- Usable in every scaling and hydrate file, the events the detector currently misses (P-TPT: 64% of class 8, 35% of class 9).
- Availability fingerprint: usable in 55% of normal files but 100% of several event types. With max-over-sensors scoring, more valid sensors means more chances of a high score by chance, so extra detections could partly come from sensor count rather than physics.
- Control when adding it: check whether false alarms rise in the normal periods of the same event files.

### Scaling file WELL-00001_20170226130146: P-TPT

- Normal: regular hourly oscillation of about ±0.04 bar (13.27M–13.35M Pa).
- Transient: steady climb of ~0.35 bar over ~9 h, about 4× normal's full range; the event ends with a sharp drop.
- Scaling visibility depends on the well: here it is plain in P-TPT; in the earlier file the creep was about the size of normal waves.
- Why it is still missed: the 1-h level change during the climb (~0.04 bar/h) is no larger than normal's ordinary swings. What stands out is the level itself, which exceeds everything this well reached in normal operation a few hours into the transient.

**Next feature (v5):** level relative to the well's own normal range (per-well tier only, no-future rule).

### Scaling file WELL-00001_20170226130146: P-MON-CKP vs P-TPT

- P-MON-CKP (just upstream of the production choke) rises ~0.32 bar over the transient, similar in size to P-TPT (~0.35 bar), but as a smooth, steady, accelerating climb without P-TPT's ±0.04 bar hourly waves: much better signal-to-noise.
- P-MON-CKP clearly exceeds its normal range around 01:30–02:00, earlier than P-TPT (around 03:00–05:00), roughly 7–8 h before the event.
- Physics: the sensor sits right behind the choke where scale builds up, so it reflects the narrowing directly.
- During the normal phase, P-MON-CKP drifts up slowly (~0.035 bar over 9 h): ordinary variation or scaling starting before the label. Unresolved from one file.

**Implications for v5**
- Add P-MON-CKP as an extended sensor; the existing 1-h level change may already detect scaling on it, given its low noise.
- Add level relative to the well's own normal range (per-well tier only, no-future rule).
- Control for the availability fingerprint: check false alarms in the normal periods of event files when P-MON-CKP is added.

### v5a: adding P-MON-CKP (v4 features, k=5)

- Reproducibility: v4_core results on the rebuilt dataset are identical to the earlier v4 run.
- Macro detection changes by about ±1 point (e.g. 0.348 → 0.356 at q=0.995); early warning unchanged. P-MON-CKP did not deliver the expected scaling detection.
- False alarms mixed: worse at q=0.995 (0.072 → 0.132/day), better at q=0.99 (0.409 → 0.331). No extra detections, so the lottery effect does not apply yet.
- Episode counts are not monotonic in the threshold: a stricter threshold can split one alarm episode into two.
- Hypothesis: one shared threshold over the max of 16 feature z-scores is set by the heaviest-tailed (noisiest) features, masking moderate but meaningful deviations on a clean sensor. Checking inside the detector on the scaling file.

In [21]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def detected_by_class(rep):
    ev = rep[rep["has_event"]].assign(detected=lambda d: d["detected"].astype(bool))
    return ev.groupby("event_class")["detected"].mean()

In [23]:
import importlib
import assetmind.ingestion.threew as threew_module
importlib.reload(threew_module)

<module 'assetmind.ingestion.threew' from 'C:\\Users\\jeffy\\LLMOPs projects\\assetmind\\src\\assetmind\\ingestion\\threew.py'>

In [26]:
from assetmind.models.anomaly_twotier import TwoTierDetector, robust_stats
from assetmind.ingestion.threew import MODEL_SENSORS_EXT
from assetmind.models.eval_anomaly_twotier import load_windows

wx, metax = load_windows(ROOT / "data/gold/3w_windows_real_ext_h.parquet")
inst, well = "7/WELL-00001_20170226130146", "WELL-00001"
V4F = ("logstd", "change", "lvl1h", "logwav1h")

det = TwoTierDetector(MODEL_SENSORS_EXT, 0.995, 5, 120, "out_of_sample", V4F).fit(
    wx[(wx["well"] != well) & (wx["phase"] == "normal")])
g = wx[wx["instance_id"] == inst]
wm = metax[metax["well"] == well]
hist = wx[wx["instance_id"].isin(wm.index[wm["end"] < metax.loc[inst, "start"]]) & (wx["phase"] == "normal")]
print("History windows:", len(hist), "| tier:", det.score_instance(g, hist)[2],
      "| per-well threshold:", round(det.well_threshold_, 2))

# z-scores of this file's windows against the well's own history
med, scale = robust_stats(det.global_._features(hist))
z = ((det.global_._features(g) - med) / scale).abs()
z_plot = z.assign(window_end=g["window_end"].values)
cols = [c for c in z.columns if c.startswith("P-MON-CKP")]
fig = px.line(z_plot.melt(id_vars="window_end", value_vars=cols, var_name="feature", value_name="z"),
              x="window_end", y="z", color="feature", title="Scaling file: P-MON-CKP feature z-scores vs threshold")
fig.add_hline(y=det.well_threshold_, line_dash="dash", annotation_text="threshold")
fig.show()

# Which features produce the largest z during NORMAL history?
z_hist = ((det.global_._features(hist) - med) / scale).abs().dropna(how="all")
print("\nFeature that is the maximum in normal windows (share):")
print(z_hist.idxmax(axis=1).value_counts(normalize=True).head(6).round(2).to_string())

History windows: 5426 | tier: well | per-well threshold: 1119.66



Feature that is the maximum in normal windows (share):
P-MON-CKP_change      0.29
P-MON-CKP_logwav1h    0.13
P-ANULAR_logstd       0.09
P-TPT_logstd          0.08
T-TPT_change          0.08
P-MON-CKP_lvl1h       0.08


In [25]:
train_normal = wx[(wx["well"] != well) & (wx["phase"] == "normal")]
parts = []
for wname, gw in train_normal.groupby("well"):
    span = gw.groupby("instance_id")["window_end"].agg(["min", "max"]).sort_values("min")
    if len(gw) < 120 or len(span) < 2:
        continue
    cut = span["min"].iloc[len(span) // 2]
    early = gw[gw["instance_id"].isin(span.index[span["max"] < cut])]
    late = gw[gw["instance_id"].isin(span.index[span["min"] >= cut])]
    if len(early) < 120 or late.empty:
        continue
    med_e, scale_e = robust_stats(det.global_._features(early))
    zc_w = ((det.global_._features(late) - med_e) / scale_e).abs()
    zc_w["well"] = wname
    parts.append(zc_w)

zc = pd.concat(parts)
feat_cols = [c for c in zc.columns if c != "well"]
top = zc[feat_cols].max(axis=1)
thr = top.quantile(0.995)
print("Recomputed threshold:", round(thr, 1))

print("\n99.5th percentile of z, per feature (top 8):")
print(zc[feat_cols].quantile(0.995).sort_values(ascending=False).head(8).round(1).to_string())

extreme = zc[top >= thr]
print("\nWhich feature is the max in the windows above the threshold:")
print(extreme[feat_cols].idxmax(axis=1).value_counts().head(5).to_string())
print("\nWhich wells they come from:")
print(extreme["well"].value_counts().head(5).to_string())

Recomputed threshold: 1119.7

99.5th percentile of z, per feature (top 8):
P-ANULAR_logwav1h    591884.5
P-ANULAR_lvl1h        40872.7
P-TPT_logwav1h         2586.9
P-ANULAR_logstd        1319.9
T-TPT_logwav1h          587.9
P-MON-CKP_lvl1h         233.0
P-TPT_lvl1h             198.3
T-TPT_lvl1h             152.2

Which feature is the max in the windows above the threshold:
T-TPT_logwav1h       173
P-ANULAR_logwav1h    146
P-ANULAR_lvl1h        80
P-ANULAR_logstd       59
P-MON-CKP_change      24

Which wells they come from:
well
WELL-00014    458
WELL-00041     24
WELL-00004     18


### Diagnosis: what sets the 1,120 threshold

- 99.5th-percentile z by feature: P-ANULAR_logwav1h ≈ 591,885; P-ANULAR_lvl1h ≈ 40,873; P-TPT_logwav1h ≈ 2,587.
- P-ANULAR's held, step-like values give 1-hour features a near-zero (but non-zero) spread in early files; a single later step yields enormous z-scores.
- WELL-00014 (the calmest well) produces 458 of ~500 windows above the threshold, so one well sets the per-well threshold for all others.
- Fix: a minimum spread per feature, a fraction (10%) of that feature's global spread, applied in both calibration and scoring.

In [27]:
importlib.reload(importlib.import_module("assetmind.models.anomaly_twotier"))
from assetmind.models.anomaly_twotier import TwoTierDetector
det_f = TwoTierDetector(MODEL_SENSORS_EXT, 0.995, 5, 120, "out_of_sample", V4F, scale_floor=0.1).fit(
    wx[(wx["well"] != well) & (wx["phase"] == "normal")])
print("Per-well threshold with floor:", round(det_f.well_threshold_, 2))

Per-well threshold with floor: 243.91


### Scale floor (10% of global spread), v4 features, k=5

- Per-well threshold on the scaling-file setup: 1,120 → 244. Better, but still far too high: the 1-hour features' global spread is itself tiny (mostly near-zero hour changes), so 10% of it remains tiny.
- v4_core: floor dominates at q=0.99 (0.437 vs 0.417 detection at the same 0.409 FA/day), but is worse at the quiet end (no-floor q=0.995: 0.348 at 0.072 FA/day; floor q=0.999: 0.303 at 0.114). The curves cross.
- Early warning moved for the first time: warn_ge_1h 0.140 → 0.159 (q=0.995), 0.178 → 0.206 (q=0.99).
- Event-file false alarms rose (0.153 → 0.517/day at q=0.995): the per-well tier is now active and noisy.
- P-MON-CKP still does not help (q=0.995: 0.337 vs 0.392 detection); at q=0.99 it raises pure-normal false alarms to 0.49/day.
- Root cause: the max of 16 z-scores compared with one threshold lets the heaviest-tailed feature set the bar. Next: per-feature percentile calibration.

In [28]:
curves = pd.concat([
    pd.read_parquet(ROOT / "reports/3w_sweep_v3_vs_v4.parquet").assign(
        version=lambda d: d["features"].map({"v3": "v3 (5-min)", "v4": "v4 (+1-h)"})),
    pd.read_parquet(ROOT / "reports/3w_sweep_floor.parquet").assign(version=lambda d: d["config"]),
    pd.read_parquet(ROOT / "reports/3w_sweep_v6.parquet").assign(version=lambda d: d["config"]),
])
fig = px.line(curves.sort_values("false_alarms_per_day"), x="false_alarms_per_day", y="detection_macro",
              color="version", markers=True, title="Scoreboard: macro detection vs false alarms per well per day")
fig.show()

### v6: per-feature percentile scoring (per-well tier), floor 0.1, v4 features, k=5

| Config | q | Macro detection | FA/day |
|---|---|---|---|
| v6_core | 0.99 | 0.469 | 0.451 |
| v6_core | 0.995 | 0.373 | 0.156 |
| v6_core | 0.999 | 0.330 | 0.150 |
| v6_ext | 0.99 | 0.454 | 0.373 |
| v6_ext | 0.995 | 0.409 | 0.228 |
| v6_ext | 0.999 | 0.325 | 0.180 |

- v6 has the best curve in the 0.2–0.45 FA/day range: ~0.455 vs v4's 0.417 at ~0.41 FA/day (~4 points, about two files' worth).
- At the quiet end, v6 cannot go below ~0.15 FA/day; raising q from 0.995 to 0.999 barely reduces false alarms. v4 (q=0.995: 0.348 at 0.072) remains the best quiet option. Suspected source: the unchanged global tier.
- P-MON-CKP now adds slightly (v6_ext above v6_core mid-range), within one-file noise.
- Noise level: one file in a small class shifts macro detection by ~2 points; differences below that are not meaningful.
- Ceiling: macro detection ≤ ~0.47, warn ≥1 h ≤ ~0.20 across all versions. Diminishing returns for the robust-z family.

In [29]:
files = {"v4": "3w_lowo_v4_core_q0.995_k5.parquet",
         "v6_core": "3w_lowo_v6_core_q0.995_k5.parquet",
         "v6_ext": "3w_lowo_v6_ext_q0.995_k5.parquet"}
reps = {k: pd.read_parquet(ROOT / "reports" / f) for k, f in files.items()}

cmp6 = pd.DataFrame({k: detected_by_class(r) for k, r in reps.items()}).round(2)
cmp6.insert(0, "files", reps["v6_core"][reps["v6_core"]["has_event"]].groupby("event_class").size())
print(cmp6.to_string())

sw6 = pd.read_parquet(ROOT / "reports/3w_sweep_v6.parquet")
print("\n", sw6[["config", "quantile", "fa_day_well", "fa_day_global"]].round(3).to_string(index=False))

             files    v4  v6_core  v6_ext
event_class                              
1                4  0.00     0.00    0.00
2               22  1.00     1.00    1.00
3               32  0.97     0.97    0.97
4              343  0.06     0.06    0.05
5               11  0.45     0.55    0.55
6                6  0.00     0.00    0.33
7               36  0.00     0.00    0.00
8               14  0.50     0.57    0.64
9               14  0.14     0.21    0.14

  config  quantile  fa_day_well  fa_day_global
v6_core     0.990        0.240          1.314
v6_core     0.995        0.150          0.183
v6_core     0.999        0.120          0.275
 v6_ext     0.990        0.348          0.510
 v6_ext     0.995        0.206          0.353
 v6_ext     0.999        0.170          0.236


### v6 per class (q=0.995) and false-alarm sources

- Quick PCK restriction: 0 → 2 of 6 detected only with P-MON-CKP (v6_ext), consistent with the sensor sitting directly upstream of the choke.
- Productivity loss +1 file; hydrate (production) +1 (v6_core) and +1 more with P-MON-CKP.
- Scaling: 0% in every version. The 1-h change features cannot see slow creep; the level-vs-own-normal feature (v5b) was never built.
- False alarms come from both tiers in v6: per-well 0.12–0.35/day, global 0.18–1.31/day.

**Baseline decision**
- Arm A anomaly baseline = Pareto front across all robust-z versions; default operating point v6_ext, q=0.995 (macro detection 0.409, 0.228 FA/well/day).
- Caveat: settings were chosen after seeing leave-one-well-out results, so the baseline is slightly optimistic (a harder bar for later models).
- Final robust-z experiment before freezing: v5b, level relative to the well's own normal range, targeting scaling.

In [30]:
v6e = pd.read_parquet(ROOT / "reports/3w_lowo_v6_ext_q0.995_k5.parquet")
v5b = pd.read_parquet(ROOT / "reports/3w_lowo_v5b_ext_q0.995_k5.parquet")
cmp_b = pd.DataFrame({"v6_ext": detected_by_class(v6e), "v5b_ext": detected_by_class(v5b)}).round(2)
cmp_b.insert(0, "files", v5b[v5b["has_event"]].groupby("event_class").size())
print(cmp_b.to_string())
print("\nScaling file detected?",
      v5b.loc[v5b["instance_id"] == "7/WELL-00001_20170226130146", ["detected", "lead_time_h", "tier"]].to_dict("records"))

             files  v6_ext  v5b_ext
event_class                        
1                4    0.00     0.00
2               22    1.00     1.00
3               32    0.97     0.97
4              343    0.05     0.08
5               11    0.55     0.55
6                6    0.33     0.50
7               36    0.00     0.00
8               14    0.64     0.64
9               14    0.14     0.14

Scaling file detected? [{'detected': False, 'lead_time_h': nan, 'tier': 'well'}]


### v5b: level relative to the well's own earlier normal (per-well tier only)

| Config | q | Macro detection | FA/day | Per-well FA/day |
|---|---|---|---|---|
| v5b_ext | 0.99 | 0.470 | 0.613 | 0.632 |
| v5b_ext | 0.995 | 0.431 | 0.451 | 0.469 |
| v5b_ext | 0.999 | 0.386 | 0.277 | 0.284 |

- Every v5b point is dominated by v6_ext or v6_core (e.g. v5b q=0.995: 0.431 at 0.451 vs v6_ext q=0.99: 0.454 at 0.373). Early warning unchanged.
- Per-well false alarms more than doubled (0.206 → 0.469/day at q=0.995): the drift risk materialized. A baseline from earlier files (weeks to months older) flags "new normal" operating levels, which are sustained, so the persistence rule cannot filter them.
- Proper fix: a recent, adaptive baseline (e.g. the last few days of normal). Not possible with 3W, whose files are disjoint snapshots. Dataset limitation; future work for continuous deployment data.
- Decision: v5b rejected. Robust-z family frozen; Arm A anomaly baseline = Pareto front of v4, v4+floor and v6; default v6_ext, q=0.995.

### v5b per class (q=0.995)

- Quick restriction: 0.33 → 0.50 (+1 file), consistent with physics: the settled restriction is a sustained level shift.
- Flow instability: 0.05 → 0.08. Scaling: still 0%, including WELL-00001_20170226130146 (per-well tier), whose P-MON-CKP rose ~0.32 bar.
- Likely reason: the level threshold is calibrated on between-file level differences (weeks to months apart), which exceed scaling's creep.
- Conclusion: level carries real signal, but against old baselines its false-alarm cost outweighs the gains. Rejected.

## Summary: robust-z anomaly detection on 3W (Arm A baseline)

**Evaluation:** leave-one-well-out on 1,119 real instances (40 wells), event-level scoring, no-future rule for per-well baselines, macro detection vs false alarms per well per day.

| Version | Change | Effect |
|---|---|---|
| v1 | Global robust z (std, change), max over sensors, k=3 | DHSV 100%, slugging 84%, instability 1% |
| v2 | Per-well baseline (no-future rule) | Better detection; per-well false alarms doubled (in-sample calibration too tight) |
| v3 | Out-of-sample calibration | False alarms fell sharply, detection fell; v2/v3 are two points on one curve |
| Sweep | Thresholds × persistence | Pareto front; persistence nearly free; ceiling ~0.47 macro detection |
| v4 | 1-h look-back features | Same detection, ~3.5× fewer false alarms at equal detection |
| v5a | + P-MON-CKP | No gain: drowned out by a shared threshold of z ≈ 1,120 |
| Floor | Minimum per-well spread | Threshold 1,120 → 244; partial fix |
| v6 | Per-feature percentile scoring | Best curve in the 0.2–0.45 FA/day range; P-MON-CKP catches 2/6 quick restrictions |
| v5b | Level vs own earlier normal | Dominated: baseline drift between files causes sustained false alarms |

**Frozen Arm A baseline:** the Pareto front of v4, v4+floor and v6; default operating point v6_ext, q=0.995, k=5 (macro detection 0.409, 0.228 FA/well/day). Selected after seeing results, so slightly optimistic.

**Key lessons**
- Inspect data before trusting it: present ≠ usable, curated vs unselected normal, fixed-length conventions.
- Compare like with like (selection effects) and against a chance baseline before claiming a pattern.
- Look inside the model on a known case: aggregate scores hid a threshold of z ≈ 1,120.
- Features limit the ceiling more than thresholds; slow events need adaptive baselines that 3W's disjoint snapshots cannot support.

**Open limitations:** scaling undetected; flow instability ≤ 8%; false-alarm rate optimistic on curated normal files.

In [31]:
curves = pd.concat([
    pd.read_parquet(ROOT / "reports/3w_sweep_v3_vs_v4.parquet").query("features == 'v4'").assign(version="v4"),
    pd.read_parquet(ROOT / "reports/3w_sweep_floor.parquet").query("config == 'v4_core_floor'").assign(version="v4_core_floor"),
    pd.read_parquet(ROOT / "reports/3w_sweep_v6.parquet").assign(version=lambda d: d["config"]),
    pd.read_parquet(ROOT / "reports/3w_sweep_v7.parquet").assign(version="v7_ext_iforest"),
])
fig = px.line(curves.sort_values("false_alarms_per_day"), x="false_alarms_per_day", y="detection_macro",
              color="version", markers=True, title="Scoreboard: frozen robust-z baseline vs Isolation Forest")
fig.show()

### v7: Isolation Forest on per-well z-vectors (same setup as v6_ext; only the combining step changed)

| q | Macro detection | FA/day | Per-well FA | Global FA | warn ≥1 h |
|---|---|---|---|---|---|
| 0.99 | 0.456 | 0.505 | 0.504 | 0.510 | 0.196 |
| 0.995 | 0.424 | 0.222 | 0.199 | 0.353 | 0.178 |
| 0.999 | 0.296 | 0.054 | 0.021 | 0.236 | 0.112 |

- Dominates v6_ext at q=0.995 (0.424 vs 0.409 detection, 0.222 vs 0.228 FA/day) and adds the quietest point on the board (0.054 FA/day at q=0.999). v6_ext is better at q=0.99.
- Best early warning in the mid range (warn ≥1 h 0.178 vs 0.150).
- Differences are mostly 1–2 points (about one file) and the forest is random (fixed seed), so v7 ≈ v6 overall. Combining features differently adds little: the bottleneck is information in the features, not the combination rule.
- At the quiet end, the global tier causes ~11× the per-well false alarms (0.236 vs 0.021/day): the next target for false-alarm reduction.

In [32]:
v7r = pd.read_parquet(ROOT / "reports/3w_lowo_v7_ext_iforest_q0.995_k5.parquet")
cmp7 = pd.DataFrame({"v6_ext": detected_by_class(v6e), "v7_iforest": detected_by_class(v7r)}).round(2)
cmp7.insert(0, "files", v7r[v7r["has_event"]].groupby("event_class").size())
print(cmp7.to_string())

             files  v6_ext  v7_iforest
event_class                           
1                4    0.00        0.25
2               22    1.00        1.00
3               32    0.97        1.00
4              343    0.05        0.15
5               11    0.55        0.45
6                6    0.33        0.00
7               36    0.00        0.03
8               14    0.64        0.64
9               14    0.14        0.29


### v7 vs v6_ext per class (q=0.995)

| Class | Files | v6_ext | v7 |
|---|---|---|---|
| 1 BSW increase | 4 | 0.00 | 0.25 |
| 2 DHSV closure | 22 | 1.00 | 1.00 |
| 3 Slugging | 32 | 0.97 | 1.00 |
| 4 Flow instability | 343 | 0.05 | 0.15 |
| 5 Productivity loss | 11 | 0.55 | 0.45 |
| 6 Quick restriction | 6 | 0.33 | 0.00 |
| 7 Scaling | 36 | 0.00 | 0.03 |
| 8 Hydrate (production) | 14 | 0.64 | 0.64 |
| 9 Hydrate (service) | 14 | 0.14 | 0.29 |

- Isolation Forest triples flow-instability detection (~+34 files, the largest class) and gives the first scaling and BSW detections (1 file each): it catches several moderately unusual features together.
- Max-percentile keeps the quick restrictions (−2 with v7): it catches one extremely unusual feature (P-MON-CKP).
- The methods are complementary; similar macro detection hides opposite per-class strengths. Micro detection rises sharply with v7 because instability is 343 of 525 event files.
- Next candidates: ensemble of percentile and forest scores; percentile/forest scoring for the global tier; LSTM autoencoder.

In [33]:
v8r = pd.read_parquet(ROOT / "reports/3w_lowo_v8_ext_ensemble_q0.995_k5.parquet")
cmp8 = pd.DataFrame({"v6_ext": detected_by_class(v6e), "v7_iforest": detected_by_class(v7r),
                     "v8_ensemble": detected_by_class(v8r)}).round(2)
cmp8.insert(0, "files", v8r[v8r["has_event"]].groupby("event_class").size())
print(cmp8.to_string())

curves = pd.concat([curves, pd.read_parquet(ROOT / "reports/3w_sweep_v8.parquet").assign(version="v8_ext_ensemble")])
fig = px.line(curves.sort_values("false_alarms_per_day"), x="false_alarms_per_day", y="detection_macro",
              color="version", markers=True, title="Scoreboard with the ensemble")
fig.show()

             files  v6_ext  v7_iforest  v8_ensemble
event_class                                        
1                4    0.00        0.25         0.00
2               22    1.00        1.00         1.00
3               32    0.97        1.00         0.97
4              343    0.05        0.15         0.06
5               11    0.55        0.45         0.55
6                6    0.33        0.00         0.33
7               36    0.00        0.03         0.00
8               14    0.64        0.64         0.64
9               14    0.14        0.29         0.36


### v8: ensemble (max of percentile and Isolation Forest ranks, calibrated once)

| q | Macro detection | FA/day | Per-well FA | Global FA | warn ≥1 h |
|---|---|---|---|---|---|
| 0.99 | 0.494 | 0.313 | 0.277 | 0.510 | 0.206 |
| 0.995 | 0.434 | 0.253 | 0.234 | 0.353 | 0.159 |
| 0.999 | 0.325 | 0.156 | 0.142 | 0.236 | 0.121 |

- q=0.99 is the best point on the scoreboard: 0.494 macro detection at 0.313 FA/day, dominating all previous sensitive settings (v6_ext q=0.99: 0.454 at 0.373), and the first result above the ~0.47 ceiling. Best early warning (0.206).
- At the quiet end the ensemble does not help (q=0.999 dominated by v7 q=0.995 and v4).
- Per class at q=0.995: keeps v6's quick restrictions (0.33) and productivity loss (0.55), best hydrate (service) yet (0.36), but loses v7's instability gain (0.06 vs 0.15) and its single BSW/scaling catches.
- Reason: calibrating on the max of two ranks raises the bar for each method; moderately unusual windows (instability for the forest) fall below it, while strong single-feature signals survive.

In [34]:
v8r_99 = pd.read_parquet(ROOT / "reports/3w_lowo_v8_ext_ensemble_q0.99_k5.parquet")
v7r_99 = pd.read_parquet(ROOT / "reports/3w_lowo_v7_ext_iforest_q0.99_k5.parquet")
v6e_99 = pd.read_parquet(ROOT / "reports/3w_lowo_v6_ext_q0.99_k5.parquet")
cmp8_99 = pd.DataFrame({"v6_ext": detected_by_class(v6e_99), "v7_iforest": detected_by_class(v7r_99),
                        "v8_ensemble": detected_by_class(v8r_99)}).round(2)
cmp8_99.insert(0, "files", v8r_99[v8r_99["has_event"]].groupby("event_class").size())
print(cmp8_99.to_string())

             files  v6_ext  v7_iforest  v8_ensemble
event_class                                        
1                4    0.00        0.25         0.25
2               22    1.00        1.00         1.00
3               32    0.97        1.00         1.00
4              343    0.09        0.22         0.19
5               11    0.55        0.55         0.55
6                6    0.33        0.00         0.33
7               36    0.00        0.08         0.06
8               14    0.79        0.64         0.71
9               14    0.36        0.36         0.36


### v8 per class at q=0.99

| Class | Files | v6_ext | v7 | v8 |
|---|---|---|---|---|
| 1 | 4 | 0.00 | 0.25 | 0.25 |
| 2 | 22 | 1.00 | 1.00 | 1.00 |
| 3 | 32 | 0.97 | 1.00 | 1.00 |
| 4 | 343 | 0.09 | 0.22 | 0.19 |
| 5 | 11 | 0.55 | 0.55 | 0.55 |
| 6 | 6 | 0.33 | 0.00 | 0.33 |
| 7 | 36 | 0.00 | 0.08 | 0.06 |
| 8 | 14 | 0.79 | 0.64 | 0.71 |
| 9 | 14 | 0.36 | 0.36 | 0.36 |

At q=0.99 the ensemble recovers most of the forest's instability gain (0.19 vs 0.22; v6 0.09) while keeping the percentile method's quick restrictions (0.33), plus BSW (1 of 4) and scaling (2 of 36). No method wins every class, but the ensemble covers the most event types.

## Updated Arm A anomaly baseline: operating points on the Pareto front

| Mode | Setting | Macro detection | FA/well/day |
|---|---|---|---|
| Quiet | v7 Isolation Forest, q=0.999 | 0.296 | 0.054 |
| Balanced | v7 Isolation Forest, q=0.995 | 0.424 | 0.222 |
| Sensitive | v8 ensemble, q=0.99 | 0.494 | 0.313 |

Caveats: settings selected after seeing leave-one-well-out results (optimistic front); v7/v8 depend on a random seed (robustness check pending).

### Isolation Forest seed check (seeds 42, 7, 123)

| Setting | Macro detection | FA/day | Instability files |
|---|---|---|---|
| v8 ensemble, q=0.99 | 0.486–0.494 | 0.313–0.349 | 64–66 |
| v7 forest, q=0.999 | 0.281–0.296 | 0.054–0.060 | 0 |
| v7 forest, q=0.995 | 0.395–0.426 | 0.222–0.241 | 49–59 |

- Sensitive (v8, q=0.99) and quiet (v7, q=0.999) points are stable; v8 stays above the old ~0.47 ceiling for every seed and keeps both quick restrictions.
- v7 at q=0.995 is seed-sensitive (~3 points): the earlier claim that it dominates v6_ext at q=0.995 does not hold for seed 123. Its instability gain over v6 (≥49 vs ~17 files) is robust.
- The ensemble is more stable than the forest alone, since its percentile half is deterministic.
- Balanced operating point changed to v6_ext, q=0.995 (deterministic, 0.409 at 0.228 FA/day).

In [35]:
curves = pd.concat([curves, pd.read_parquet(ROOT / "reports/3w_sweep_v9.parquet").assign(version="v9_ext_lstm")])
px.line(curves.sort_values("false_alarms_per_day"), x="false_alarms_per_day", y="detection_macro",
        color="version", markers=True, title="Scoreboard with the LSTM autoencoder").show()

v9r = pd.read_parquet(ROOT / "reports/3w_lowo_v9_ext_lstm_q0.99_k5.parquet")
cmp9 = pd.DataFrame({"v8_ensemble": detected_by_class(v8r_99), "v9_lstm": detected_by_class(v9r)}).round(2)
cmp9.insert(0, "files", v9r[v9r["has_event"]].groupby("event_class").size())
print(cmp9.to_string())

             files  v8_ensemble  v9_lstm
event_class                             
1                4         0.25     0.75
2               22         1.00     1.00
3               32         1.00     1.00
4              343         0.19     0.43
5               11         0.55     0.45
6                6         0.33     0.83
7               36         0.06     0.06
8               14         0.71     0.57
9               14         0.36     0.14


### v9: LSTM autoencoder on 30-window sequences of per-well z-scores (first run)

| q | Macro detection | FA/day | Per-well FA | warn ≥1 h |
|---|---|---|---|---|
| 0.99 | 0.582 | 1.395 | 1.555 | 0.187 |
| 0.995 | 0.556 | 1.022 | 1.143 | 0.178 |
| 0.999 | 0.526 | 0.517 | 0.568 | 0.159 |

- Highest detection so far, but in a high-false-alarm region; curves barely overlap with earlier methods.
- At ~0.5 FA/day: v9 (q=0.999) 0.526 vs v7 (q=0.99) 0.456 macro detection, about +7 points (3–4 files), beyond one-file noise.
- High false alarms because consecutive LSTM scores share 29 of 30 windows: exceedances last long, so the k=5 persistence filter barely helps.
- Next: stricter thresholds (q up to 0.9999) using cached fold scores, then compare at equal false alarms; seed check if the result is close.

In [36]:
sw9 = pd.read_parquet(ROOT / "reports/3w_sweep_v9.parquet")
print(sw9[["quantile", "detection_macro", "false_alarms_per_day", "warn_ge_1h"]].round(3).to_string(index=False))

v9r = pd.read_parquet(ROOT / "reports/3w_lowo_v9_ext_lstm_q0.999_k5.parquet")
v7r_99 = pd.read_parquet(ROOT / "reports/3w_lowo_v7_ext_iforest_q0.99_k5.parquet")
cmp9 = pd.DataFrame({"v7_iforest_q0.99": detected_by_class(v7r_99),
                     "v9_lstm_q0.999": detected_by_class(v9r)}).round(2)
cmp9.insert(0, "files", v9r[v9r["has_event"]].groupby("event_class").size())
print(cmp9.to_string())

curves = pd.concat([curves[curves["version"] != "v9_ext_lstm"], sw9.assign(version="v9_ext_lstm")])
px.line(curves.sort_values("false_alarms_per_day"), x="false_alarms_per_day", y="detection_macro",
        color="version", markers=True, title="Scoreboard with the LSTM autoencoder (extended thresholds)").show()

 quantile  detection_macro  false_alarms_per_day  warn_ge_1h
    0.990            0.582                 1.395       0.187
    0.995            0.556                 1.022       0.178
    0.999            0.526                 0.517       0.159
    1.000            0.475                 0.337       0.150
    1.000            0.444                 0.343       0.131
             files  v7_iforest_q0.99  v9_lstm_q0.999
event_class                                         
1                4              0.25            0.75
2               22              1.00            1.00
3               32              1.00            1.00
4              343              0.22            0.33
5               11              0.55            0.36
6                6              0.00            0.67
7               36              0.08            0.06
8               14              0.64            0.43
9               14              0.36            0.14


### v9 LSTM autoencoder: extended thresholds (cached fold scores)

| q | Macro detection | FA/day | Per-well FA | warn ≥1 h |
|---|---|---|---|---|
| 0.99 | 0.582 | 1.395 | 1.555 | 0.187 |
| 0.995 | 0.556 | 1.022 | 1.143 | 0.178 |
| 0.999 | 0.526 | 0.517 | 0.568 | 0.159 |
| 0.9995 | 0.475 | 0.337 | 0.391 | 0.150 |
| 0.9999 | 0.444 | 0.343 | 0.398 | 0.131 |

- In the 0.3–0.4 FA/day range the ensemble wins (v8 q=0.99: 0.494 at 0.313 vs v9 q=0.9995: 0.475 at 0.337). Above ~0.4 FA/day the LSTM is best (0.526 at 0.517, up to 0.58).
- False-alarm floor of ~0.34/day: stricter thresholds stop reducing false alarms (q=0.9995 → 0.9999), as some normal stretches produce errors beyond any threshold.
- Per class at ~0.5 FA/day vs v7 (q=0.99): quick restriction 4/6 (vs 0/6), BSW 3/4 (vs 1/4), instability +~38 files; weaker on hydrates (−3 each), productivity loss (−2) and scaling (−1).
- The LSTM excels on events with a characteristic temporal shape within 30 minutes; slow, multi-day events look normal inside its memory.
- Seed check pending (small per-class counts).

### Final Arm A operating points

| Mode | Setting | Macro detection | FA/well/day |
|---|---|---|---|
| Quiet | v7 forest, q=0.999 | 0.28–0.30 | 0.05–0.06 |
| Balanced | v6_ext percentile, q=0.995 | 0.409 | 0.228 |
| Sensitive | v8 ensemble, q=0.99 | 0.49 | 0.31–0.35 |
| Very sensitive | v9 LSTM, q=0.999 | 0.526 | 0.517 |